# 01 — Field Overview

**Plots:** sky positions · Gaia CMD · HST CMDs · image footprints

Edit the `CONFIG` cell below, then run all cells.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = '..'          # pipeline root (contains the field subdirectory)
FIELD_NAME  = 'Sculptor_dSph'
TELESCOPE   = 'HST'
IM_TYPE     = '_flc'
# ─────────────────────────────────────────────────────────────────────────────

import sys, os
# bp3m is installed as a package — no path manipulation needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_bp3m_results, load_cross_match_results,
    load_transformations, sky_extent,
)

DATA = Path(OUTPUT_DIR)
print(f'Field: {FIELD_NAME}')

In [ ]:
# Load data
_gaia_files = sorted((DATA / FIELD_NAME / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])
print(f'Gaia stars: {len(gaia)}')

try:
    bp3m = load_bp3m_results(DATA / FIELD_NAME / 'BP3M_results')
    stars = bp3m['stars']
    print(f'BP3M stars: {len(stars)}')
except FileNotFoundError:
    bp3m = None
    stars = None
    print('BP3M results not found — some plots will be skipped.')

try:
    xm = load_cross_match_results(DATA, FIELD_NAME, TELESCOPE, IM_TYPE)
    tran = load_transformations(DATA, FIELD_NAME, TELESCOPE, IM_TYPE)
    print(f'Cross-matched images: {len(tran)}')
except FileNotFoundError:
    xm = tran = None
    print('Cross-match results not found.')

In [ ]:
# ── Sky positions ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(gaia['ra'], gaia['dec'], c=gaia['gmag'],
                cmap='viridis_r', s=3, alpha=0.6, rasterized=True)
plt.colorbar(sc, ax=ax, label='Gaia G (mag)')

# Overlay image footprints from transformation.csv
if tran is not None:
    from astropy.wcs import WCS
    for img_name, row in tran.iterrows():
        try:
            ra0, dec0 = float(row['ra_cen']), float(row['dec_cen'])
            pscale = float(row['pixel_scale']) / 3600.0  # deg/pix
            # Approximate footprint: ±2048×2048 pixels from centre
            half = 2048 * pscale
            rect = mpatches.Rectangle(
                (ra0 + half/np.cos(np.deg2rad(dec0)), dec0 - half),
                -2 * half / np.cos(np.deg2rad(dec0)), 2 * half,
                linewidth=0.7, edgecolor='red', facecolor='none', alpha=0.5)
            ax.add_patch(rect)
        except Exception:
            pass

(ra_lim, dec_lim) = sky_extent(gaia['ra'].values, gaia['dec'].values)
ax.set_xlim(*ra_lim)
ax.set_ylim(*dec_lim)
ax.set_xlabel('R.A. (deg)')
ax.set_ylabel('Dec. (deg)')
ax.set_title(f'{FIELD_NAME} — sky positions')
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_01_sky_positions.png', dpi=150)
plt.show()

In [ ]:
# ── Gaia colour–magnitude diagram ─────────────────────────────────────────────
has_color = np.isfinite(gaia['bp_rp'].values) if 'bp_rp' in gaia.columns else np.zeros(len(gaia), bool)

# Identify which Gaia stars have HST matches
matched_ids = set()
if xm is not None and 'gaia_source_id' in xm.columns:
    matched_ids = set(xm['gaia_source_id'].unique())
gaia_matched = gaia['source_id'].isin(matched_ids) if 'source_id' in gaia.columns else np.zeros(len(gaia), bool)

fig, ax = plt.subplots(figsize=(5, 6))
ax.scatter(gaia.loc[has_color & ~gaia_matched, 'bp_rp'],
           gaia.loc[has_color & ~gaia_matched, 'gmag'],
           c='steelblue', s=2, alpha=0.3, rasterized=True, label='All Gaia')
ax.scatter(gaia.loc[has_color & gaia_matched, 'bp_rp'],
           gaia.loc[has_color & gaia_matched, 'gmag'],
           c='darkorange', s=6, alpha=0.7, rasterized=True,
           label=f'HST-matched ({gaia_matched.sum()})', zorder=3)
ax.invert_yaxis()
ax.set_xlabel('BP − RP (mag)')
ax.set_ylabel('Gaia G (mag)')
ax.legend(loc='upper left', fontsize=8)
ax.set_title(f'{FIELD_NAME} — Gaia CMD')
plt.tight_layout()
plt.savefig(DATA / FIELD_NAME / 'plots_01_gaia_cmd.png', dpi=150)
plt.show()

In [ ]:
# ── HST colour–magnitude diagrams (G vs G−HST per filter) ─────────────────────
if xm is not None:
    # Merge matched stars with Gaia to get colour info
    merged = xm.merge(gaia[['source_id', 'gmag', 'bp_rp']].rename(
        columns={'source_id': 'gaia_source_id'}), on='gaia_source_id', how='left')

    hst_mag_col = next((c for c in ['hst_mag_st_gdc', 'hst_mag_gdc', 'hst_mag']
                        if c in merged.columns), None)

    if hst_mag_col:
        merged['color_hst'] = merged['gmag'] - merged[hst_mag_col]

        # Attach filter label from tran (indexed by image_name)
        if tran is not None and 'filter' in tran.columns:
            merged['filter'] = merged['image_name'].map(tran['filter'])
        else:
            merged['filter'] = merged['image_name']

        ok = np.isfinite(merged['color_hst']) & np.isfinite(merged['gmag'])
        filters = sorted(merged.loc[ok, 'filter'].dropna().unique())
        cmap = plt.get_cmap('tab10', len(filters))
        filt_to_color = {f: cmap(i) for i, f in enumerate(filters)}

        fig, ax = plt.subplots(figsize=(5, 6))
        for filt in filters:
            mask = ok & (merged['filter'] == filt)
            ax.scatter(merged.loc[mask, 'color_hst'], merged.loc[mask, 'gmag'],
                       s=2, alpha=0.4, color=filt_to_color[filt],
                       rasterized=True, label=filt)
        ax.invert_yaxis()
        ax.set_xlabel(f'Gaia G − {hst_mag_col} (mag)')
        ax.set_ylabel('Gaia G (mag)')
        ax.legend(fontsize=8, markerscale=3, loc='best')
        ax.set_title(f'{FIELD_NAME} — G vs G−HST CMD')
        plt.tight_layout()
        plt.savefig(DATA / FIELD_NAME / 'plots_01_hst_cmd.png', dpi=150)
        plt.show()
    else:
        print(f'No HST mag column found. Available: {list(xm.columns)}')
else:
    print('Cross-match data needed for HST CMD.')